# Cyclic peptide design: head-to-tail, disulfide, and isopeptide macrocycles

Macrocyclic peptides are constrained rings, and RFD4-Proteina designs them by being told
which atoms are bonded. This tutorial covers the three CP-SEA cyclization chemistries:

| Chemistry | Closing bond | Ideal distance |
|---|---|---|
| Head-to-tail | C-terminal `C` to N-terminal `N` (an amide) | 1.33 A |
| Disulfide | `CYS.SG` to `CYS.SG` | 2.05 A |
| Isopeptide | `LYS.NZ` to `ASP.CG` (an amide) | 1.33 A |

**Goal.** Author an input whose closure bond the model can actually see, generate designs
against it, and measure whether the ring closed.

There is one rule that decides whether a crosslink works at all, and it is a property of the
*input*, not of the checkpoint. Section 1 explains it; skipping it is the difference between a
closed ring and a pair of sulfur atoms sitting inside each other.

## Setup

Locate the repository, pick an output directory, and import what the notebook needs. Every
path takes an environment-variable override so the notebook runs from anywhere in the tree.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*NVML.*")
import logging
logging.getLogger("atomworks.constants").setLevel(logging.ERROR)

import os
import subprocess
from pathlib import Path

import numpy as np

# Locate the repository by walking up to the `.project-root` marker.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / ".project-root").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
REPO_ROOT = Path(os.environ.get("PROJECT_ROOT", REPO_ROOT))

FIXTURES = REPO_ROOT / "tests" / "data" / "cyclic_peptides"
WORK = Path(os.environ.get("CYCLIC_WORK", REPO_ROOT / "runs" / "cyclic_tutorial"))
INPUTS = WORK / "inputs"
INPUTS.mkdir(parents=True, exist_ok=True)

# AtomWorks warns when these optional mirrors are unset; nothing here needs them.
os.environ.setdefault("ALLOW_BIOTITE_CCD", "True")

print("repository :", REPO_ROOT)
print("fixtures   :", FIXTURES, "->", sorted(p.name for p in FIXTURES.glob("*.cif.gz")))
print("work dir   :", WORK)

The three fixtures are real CP-SEA macrocycles shipped with the repository, one per chemistry.
They are small (5 residues), which keeps this tutorial fast; the last section scales up to a
natural 14-residue macrocycle.

In [ ]:
import gzip

# Work on plain CIFs so the files are easy to inspect by eye.
for gz in sorted(FIXTURES.glob("*.cif.gz")):
    (INPUTS / gz.stem).write_bytes(gzip.decompress(gz.read_bytes()))

HEAD_TAIL = INPUTS / "cpsea_head_tail_4djs.cif"
DISULFIDE = INPUTS / "cpsea_disulfide_7bob.cif"
ISOPEPTIDE = INPUTS / "cpsea_isopeptide_6rlv.cif"

# The closure is declared as a _struct_conn record. This is what the parser reads.
print([ln for ln in HEAD_TAIL.read_text().splitlines() if "_struct_conn." in ln][:8])

<a id="section-1"></a>
## 1. How the model sees a bond

RFD4-Proteina's pair track carries five `[n_token, n_token]` features, computed by
`compute_token_bond_features`: `bond_mask`, `bond_order`, `is_aromatic`,
`token_bond_path_length`, and `canonical_polymer_ordering`. Two consequences drive everything
below.

**They are purely topological.** None of them encodes a target bond *length*. The model infers
geometry from what the bond pattern implies, not from a distance you supply.

**They are per token, and a standard amino acid is one token.** So a crosslink between two
standard residues arrives as "residue A is bonded to residue B" with the `SG`-`SG` or
`NZ`-`CG` atom identity *dropped*.

Head-to-tail escapes this. A backbone amide between two canonical residues is recognised as a
polymer linkage and gets `canonical_polymer_ordering = +-1`, which implies the exact atom pair
(upstream `C` to downstream `N`) and inherits the amide prior the model learned from every
peptide bond in training. It is then suppressed from `bond_mask`, because the connectivity is
already carried by the ordering feature.

Run the features on all three fixtures and the split is visible:

In [ ]:
from atomworks.constants import STANDARD_AA
from atomworks.io import parse
from atomworks.ml.utils.token import get_token_starts
from rfproteina.datapipes.bonds import compute_token_bond_features
from rfproteina.utils.parse_presets import DISTILLATION


def load_for_features(path):
    """Parse a CIF and apply the two annotations the featurizer expects."""
    aa = parse(str(path), config=DISTILLATION)["asym_unit"][0]
    aa.set_annotation("atomize", ~np.isin(aa.res_name, list(STANDARD_AA)))
    aa.set_annotation("insertion_source_idx", np.full(aa.array_length(), -1, dtype=np.int32))
    return aa


def token_of(aa, res_name, res_id):
    starts = get_token_starts(aa)
    hits = np.flatnonzero((aa.res_name[starts] == res_name) & (aa.res_id[starts] == res_id))
    assert len(hits) == 1, f"{res_name}{res_id} matched {len(hits)} tokens"
    return int(hits[0])


CASES = [
    ("head-to-tail", HEAD_TAIL, ("THR", 183), ("ASN", 179)),
    ("disulfide",    DISULFIDE, ("CYS", 118), ("CYS", 123)),
    ("isopeptide",   ISOPEPTIDE, ("LYS", 899), ("ASP", 903)),
]

print(f"{'chemistry':<14}{'ordering':>10}{'bond_mask':>11}{'path_len':>10}")
for label, path, up, dn in CASES:
    aa = load_for_features(path)
    feats = compute_token_bond_features(aa)
    u, d = token_of(aa, *up), token_of(aa, *dn)
    print(f"{label:<14}{int(feats['canonical_polymer_ordering'][d, u]):>10}"
          f"{str(bool(feats['bond_mask'][u, d])):>11}"
          f"{int(feats['token_bond_path_length'][u, d]):>10}")

Head-to-tail gets `ordering = 1` and is suppressed from `bond_mask`. Both crosslinks get
`ordering = 0` and remain visible in `bond_mask` — the model knows the two *residues* are
bonded, but not through which atoms.

**The fix is to atomize the crosslinked residues.** An atomized residue contributes one token
*per atom*, so the `SG`-`SG` edge becomes an edge between two atom tokens and the atom identity
survives. This is a supported input-side choice: `AtomizeByCCDName` honours a pre-set `atomize`
flag regardless of its own rules, training applies `RandomAtomizeResidues`, so atomized
standard residues are in-distribution, and `atomize` is a `StandardAnnotation`, so it
round-trips through an annotated CIF.

> [!IMPORTANT]
> **Atomization needs input coordinates, so it applies to template-based design only.**
> At inference every atomized token must carry a finite coordinate. In pure de novo design
> the input coordinates are all `nan`, so atomizing those residues makes generation fail
> outright. Everything in this tutorial starts from a real structure, which is why it works
> here. For de novo cyclic design, head-to-tail is the chemistry that works today;
> the crosslinks have no equivalent fix at present.

How much atomization buys depends on the chemistry and on ring size. Isopeptide needs it
unconditionally: without it the model places `LYS.NZ` against the carboxyl *oxygen* `OD1`
rather than forming the `NZ`-`CG` amide. Disulfide needs it on tight rings — the 5-residue
fixture below collapses to ~0.9 A without it — but on a realistic macrocycle the `SG`-`SG`
bridge closes either way, and atomization mainly narrows the spread. Setting it on both
crosslink chemistries is the safe default: it never costs anything.


## 2. Authoring the inputs

Three things to get right:

1. **`conn_type_id` must be `covale`.** The parse preset reads `covale` and `metalc` only. An
   RCSB entry types a disulfide as `disulf`, which is **silently dropped** — you get no error,
   just a peptide with no closure.
2. **Atomize the crosslinked residues** (`atomize = True`) for disulfide and isopeptide.
   Head-to-tail needs no atomization. Isopeptide fails without it at any size; disulfide
   fails without it on tight rings and merely scatters more on larger ones.
3. **Fix the sequence with `C_SEQ`** if the chemistry depends on residue identity. Without it
   the model may redesign `CYS`/`LYS`/`ASP` away and there is nothing left to crosslink.

In [ ]:
import rfproteina.conditions  # noqa: F401 -- register condition classes before parsing
from atomworks.io.utils.atom_array_plus import as_atom_array_plus
from atomworks.io.utils.io_utils import to_cif_file
from atomworks.ml.conditions import C_SEQ
from rfproteina.utils.parse_presets import ANNOTATED_CIF


def author(src, out_name, atomize_res_ids=(), fix_sequence=True):
    """Write an annotated CIF, optionally atomizing the crosslinked residues."""
    aa = as_atom_array_plus(parse(str(src), config=DISTILLATION)["assemblies"]["1"][0])

    if fix_sequence:
        C_SEQ.set_annotation_from_ground_truth(aa, np.ones(aa.array_length(), dtype=bool))

    categories = aa.get_annotation_categories()
    atomize = aa.atomize.copy() if "atomize" in categories else np.zeros(aa.array_length(), bool)
    if len(atomize_res_ids):
        atomize |= np.isin(aa.res_id, list(atomize_res_ids))
    aa.set_annotation("atomize", atomize)

    out = INPUTS / out_name
    to_cif_file(aa, str(out), save_standard_annotations=True, chain_disambiguation="chain_iid")

    # Round-trip: confirm the flag and the closure bond both survived.
    back = as_atom_array_plus(parse(str(out), config=ANNOTATED_CIF)["assemblies"]["1"][0])
    n_atomized = int(back.atomize.sum()) if "atomize" in back.get_annotation_categories() else 0
    print(f"{out.name}: atoms={back.array_length()} bonds={len(back.bonds.as_array())} "
          f"atomized={n_atomized}")
    return out


HT_IN = author(HEAD_TAIL, "head_tail.cif")
SS_IN = author(DISULFIDE, "disulfide_atomized.cif", atomize_res_ids=(118, 123))
IP_IN = author(ISOPEPTIDE, "isopeptide_atomized.cif", atomize_res_ids=(899, 903))

A 5-residue, 88-atom head-to-tail peptide round-trips with **88 bonds**. A linear chain of 88
atoms would have 87 — the extra one is the closure.

Validate every authored input before spending GPU time on it. Watch specifically for
`struct_conn_bond_missing`, which is how a dropped closure shows up:

In [ ]:
checker = REPO_ROOT / "skills" / "rfd4-proteina-annotated-cif" / "scripts" / "check_annotated_cif.sh"
for path in (HT_IN, SS_IN, IP_IN):
    result = subprocess.run(["bash", str(checker), str(path)], cwd=REPO_ROOT,
                            capture_output=True, text=True)
    tail = [ln for ln in result.stdout.splitlines() if "struct_conn" in ln or "error(s)" in ln]
    print(f"{path.name}:")
    for ln in tail:
        print("   ", ln.strip())

## 3. Generate

One command per chemistry. `num_replicates` is small here so the tutorial finishes quickly;
raise it for real work. Use a fresh `out_dir` whenever you change settings, so existing
structures are never silently reused.

These cells need a GPU. The rest of the notebook up to here runs on CPU.

In [ ]:
def generate(input_path, run_name, num_replicates=8, seed=0, **overrides):
    """Run rfd4-generate with Hydra key=value arguments and return the output directory."""
    out_dir = WORK / run_name
    cmd = [
        "rfd4-generate",
        f"inputs={input_path}",
        f"num_replicates={num_replicates}",
        f"out_dir={out_dir}",
        f"seed={seed}",
        "batch_size=1",
        "compress_outputs=false",
    ] + [f"{k}={v}" for k, v in overrides.items()]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)
    return out_dir


HT_RUN = generate(HT_IN, "head_tail")
SS_RUN = generate(SS_IN, "disulfide_atomized")
IP_RUN = generate(IP_IN, "isopeptide_atomized")

## 4. Measure the ring

The check is the distance between the two bonded atoms in each generated structure.

A threshold needs a **lower** bound as well as an upper one. An un-atomized disulfide typically
lands near 1.2 A — which passes a naive "closed if under 2.3 A" test *from below*, while being
physically impossible: that is well inside a single sulfur atom. A one-sided metric reports
that as success.

In [ ]:
import glob
import biotite.structure as struc
import biotite.structure.io.pdbx as pdbx

# (ideal distance, tolerance) per chemistry; closed means |d - ideal| <= tolerance.
IDEAL = {"head_tail": (1.33, 0.25), "disulfide": (2.05, 0.35), "isopeptide": (1.33, 0.25)}


def closure_distances(run_dir, pair):
    """Distance between the two bonded atoms in every generated structure of a run."""
    (res_a, atom_a), (res_b, atom_b) = pair
    out = []
    for path in sorted(glob.glob(str(Path(run_dir) / "inference_samples" / "**" / "*.cif"),
                                 recursive=True)):
        arr = pdbx.get_structure(pdbx.CIFFile.read(path), model=1)
        arr = arr[struc.filter_amino_acids(arr)]
        def coord(res_id, atom_name):
            sel = (arr.res_id == res_id) & (arr.atom_name == atom_name)
            return arr.coord[sel][0] if sel.sum() == 1 else None
        a, b = coord(res_a, atom_a), coord(res_b, atom_b)
        out.append(float(np.linalg.norm(a - b)) if a is not None and b is not None else np.nan)
    return np.array(out)


def report(label, distances, chemistry):
    ideal, tol = IDEAL[chemistry]
    finite = distances[np.isfinite(distances)]
    # Unmeasurable designs stay in the denominator.
    closed = int((np.abs(finite - ideal) <= tol).sum())
    median = f"{np.median(finite):.3f}" if len(finite) else "n/a"
    print(f"{label:<22} closed {closed}/{len(distances)}   median {median} A   ideal {ideal} A")


report("head-to-tail", closure_distances(HT_RUN, ((179, "N"), (183, "C"))), "head_tail")
report("disulfide (atomized)", closure_distances(SS_RUN, ((118, "SG"), (123, "SG"))), "disulfide")
report("isopeptide (atomized)", closure_distances(IP_RUN, ((899, "NZ"), (903, "CG"))), "isopeptide")

Read the closure against the model's *own* bond lengths, not only against the textbook value.
Short peptides come out slightly compressed overall, so the honest comparison for a head-to-tail
closure is the ordinary sequential `C(i)`-`N(i+1)` distance in the same structure:

In [ ]:
def peptide_bond_scale(run_dir):
    """Median ordinary C(i)-N(i+1) distance across a run, for use as the local reference."""
    lengths = []
    for path in sorted(glob.glob(str(Path(run_dir) / "inference_samples" / "**" / "*.cif"),
                                 recursive=True)):
        arr = pdbx.get_structure(pdbx.CIFFile.read(path), model=1)
        arr = arr[struc.filter_amino_acids(arr)]
        res_ids = arr.res_id[struc.get_residue_starts(arr)]
        for lo, hi in zip(res_ids[:-1], res_ids[1:]):
            c = arr.coord[(arr.res_id == lo) & (arr.atom_name == "C")]
            n = arr.coord[(arr.res_id == hi) & (arr.atom_name == "N")]
            if len(c) == 1 and len(n) == 1:
                lengths.append(float(np.linalg.norm(c[0] - n[0])))
    return float(np.median(lengths))


print(f"ordinary C-N in this run : {peptide_bond_scale(HT_RUN):.3f} A")
print(f"head-to-tail closure     : {np.median(closure_distances(HT_RUN, ((179, 'N'), (183, 'C')))):.3f} A")

### The control that makes this evidence

A small peptide has its termini near each other anyway. To show the *bond* closed the ring,
run the same input with the `_struct_conn` record removed and compare:

In [ ]:
import re

# Same peptide, closure record stripped.
nobond = INPUTS / "head_tail_nobond.cif"
nobond.write_text(re.sub(r"_struct_conn\.[^\n]*\n", "", HEAD_TAIL.read_text()))

NOBOND_RUN = generate(nobond, "head_tail_nobond")
report("head-to-tail (no bond)",
       closure_distances(NOBOND_RUN, ((179, "N"), (183, "C"))), "head_tail")

## 5. A realistic macrocycle

The fixtures above are 5 residues. SFTI-1 (PDB `1SFI`, chain `I`) is a natural 14-residue
sunflower trypsin inhibitor that is head-to-tail cyclic **and** disulfide-bridged, so it
exercises both chemistries at once with experimental distances to compare against:
`GLY1.N`-`ASP14.C` at 1.334 A and `CYS3.SG`-`CYS11.SG` at 2.050 A.

Note its disulfide is typed `disulf` in the RCSB entry. Left alone it is dropped silently, so
the bond must be added explicitly — the single most common way a crosslink goes missing.

This section downloads from RCSB; skip it if the machine is offline.

In [ ]:
import urllib.request

from biotite.structure import BondType
from rfproteina.utils.parse_presets import RCSB_WITH_METALS

sfti_src = INPUTS / "1SFI.cif"
if not sfti_src.exists():
    urllib.request.urlretrieve("https://files.rcsb.org/download/1SFI.cif", sfti_src)

parsed = parse(str(sfti_src), config=RCSB_WITH_METALS)
aa = as_atom_array_plus(parsed["assemblies"]["1"][0])

# Keep only the macrocycle chain. The preset exposes label_asym_id, where SFTI-1 is
# chain "B" (the entry's author chain "I"); trypsin is chain "A".
aa = aa[aa.chain_id == "B"]
assert len(set(aa.res_id.tolist())) == 14, "expected the 14-residue SFTI-1 chain"


def atom_index(res_id, atom_name):
    hits = np.flatnonzero((aa.res_id == res_id) & (aa.atom_name == atom_name))
    assert len(hits) == 1, f"{res_id}.{atom_name} matched {len(hits)} atoms"
    return int(hits[0])


# Declare both closures explicitly; the RCSB 'disulf' record is not read by the preset.
aa.bonds.add_bond(atom_index(14, "C"), atom_index(1, "N"), BondType.SINGLE)
aa.bonds.add_bond(atom_index(3, "SG"), atom_index(11, "SG"), BondType.SINGLE)

C_SEQ.set_annotation_from_ground_truth(aa, np.ones(aa.array_length(), dtype=bool))
aa.set_annotation("atomize", np.isin(aa.res_id, [3, 11]))  # the disulfide pair only

sfti_in = INPUTS / "sfti1.cif"
to_cif_file(aa, str(sfti_in), save_standard_annotations=True, chain_disambiguation="chain_iid")
print("residues:", len(set(aa.res_id.tolist())), "| atoms:", aa.array_length())

SFTI_RUN = generate(sfti_in, "sfti1")
report("SFTI-1 head-to-tail",
       closure_distances(SFTI_RUN, ((1, "N"), (14, "C"))), "head_tail")
report("SFTI-1 disulfide",
       closure_distances(SFTI_RUN, ((3, "SG"), (11, "SG"))), "disulfide")

## Checklist

- Declare every closure as `_struct_conn` with `conn_type_id = covale`. RCSB `disulf` records
  are read by nobody and fail silently.
- Validate with `check_annotated_cif.sh` and look for `struct_conn_bond_missing` before
  generating.
- Set `atomize = True` on crosslinked residues (disulfide, isopeptide). Head-to-tail does not
  need it, and neither does de novo design of any chemistry — atomized tokens need real input
  coordinates.
- Fix `C_SEQ` whenever the chemistry depends on residue identity.
- Measure closure with a two-sided threshold, and keep failures in the denominator.
- Run the no-bond control. Without it, "the termini are close" is not evidence that the ring
  closed.